In [1]:
!pip install -U huggingface_hub
import copy
import os
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from huggingface_hub import HfApi, hf_hub_download, login, notebook_login
from PIL import Image, UnidentifiedImageError
from timm.models.layers import DropPath, trunc_normal_
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchvision import datasets, transforms
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.0/521.0 kB 9.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.4.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
tokenizers 0.21.2 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.2.2 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.4 which is incompatible.
transformers 4.53.3 requires huggingface-hub<1.0,>=0.30.0, but you have huggingface-hub 1.2.2 which is incompatible.


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
TRAIN_DIR = './StyleClassificationIndoors/StyleClassificationIndoors/train' # Update if your path is different
TEST_DIR = './StyleClassificationIndoors/StyleClassificationIndoors/test'
IMG_SIZE = 224
CLASSES = {
    'asian': 0, 'boho': 1, 'coastal': 2, 'contemporary': 3, 'craftsman': 4,
    'eclectic': 5, 'farmhouse': 6, 'french-country': 7, 'industrial': 8,
    'mediterranean': 9, 'minimalist': 10, 'modern': 11, 'scandinavian': 12,
    'shabby-chic-style': 13, 'southwestern': 14, 'tropical': 15, 'victorian': 16
}

In [3]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)), 
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

val_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_tf)
val_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=val_tf)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
indices = torch.randperm(len(train_dataset)).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)

train_targets = [train_subset.dataset.targets[i] for i in train_subset.indices]
class_counts = Counter(train_targets)
class_weights = {cls: 1.0/count for cls, count in class_counts.items()}
sample_weights = [class_weights[t] for t in train_targets]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_subset,
    batch_size=32,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(val_subset, batch_size=32, shuffle=False, num_workers=2)

# features, labels = next(iter(train_loader))
# print(f"Batch Shape: {features.shape}") 
# print(f"Labels Shape: {labels.shape}") # Use .shape, not .size (which is a method)
# print(f"Unique labels in this batch: {torch.unique(labels)}") # Check if we have variety

In [4]:
def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=dilation, groups=groups, bias=False, dilation=dilation)

def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

In [5]:
class BasicBlock(nn.Module):
    """Standard Residual Block for ResNet-18 and ResNet-34"""
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, dilation=1, norm_layer=None):
        super(BasicBlock, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

In [6]:
class Bottleneck(nn.Module):
    """Bottleneck Block for ResNet-50, 101, 152"""
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, dilation=1, norm_layer=None):
        super(Bottleneck, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        width = int(planes * (base_width / 64.)) * groups
        
        self.conv1 = conv1x1(inplanes, width)
        self.bn1 = norm_layer(width)
        self.conv2 = conv3x3(width, width, stride, groups, dilation)
        self.bn2 = norm_layer(width)
        self.conv3 = conv1x1(width, planes * self.expansion)
        self.bn3 = norm_layer(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

In [7]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000, norm_layer=None):
        super(ResNet, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        self._norm_layer = norm_layer

        self.inplanes = 64
        self.dilation = 1
        self.groups = 1
        self.base_width = 64
        
        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = norm_layer(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        norm_layer = self._norm_layer
        downsample = None
        
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, self.groups,
                            self.base_width, self.dilation, norm_layer))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups,
                                base_width=self.base_width, dilation=self.dilation,
                                norm_layer=norm_layer))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x



In [8]:
def get_resnet_scratch(size='50', num_classes=17, pretrained=True):
    """
    Instantiates our scratch ResNet and loads state_dict from official sources.
    """
    configs = {
        '18':  {'block': BasicBlock, 'layers': [2, 2, 2, 2], 'weights': models.ResNet18_Weights.IMAGENET1K_V1},
        '34':  {'block': BasicBlock, 'layers': [3, 4, 6, 3], 'weights': models.ResNet34_Weights.IMAGENET1K_V1},
        '50':  {'block': Bottleneck, 'layers': [3, 4, 6, 3], 'weights': models.ResNet50_Weights.IMAGENET1K_V2}, 
        '101': {'block': Bottleneck, 'layers': [3, 4, 23, 3], 'weights': models.ResNet101_Weights.IMAGENET1K_V2},
    }
    
    cfg = configs[str(size)]
    model = ResNet(cfg['block'], cfg['layers'], num_classes=1000)
    
    if pretrained:
        print(f"Downloading and loading ResNet{size} weights...")
        state_dict = cfg['weights'].get_state_dict(progress=True)
        model.load_state_dict(state_dict)
        print("Weights loaded successfully.")

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    
    nn.init.trunc_normal_(model.fc.weight, std=.02)
    nn.init.constant_(model.fc.bias, 0)
    
    return model



In [9]:
def train_eval_split(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=5):
    """Standard training loop with validation to find optimal epoch and size"""
    best_acc = 0.0
    best_epoch = 0
    
    print(f"Starting training for {epochs} epochs...")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        
        model.eval()
        correct = 0
        total = 0
        val_loss = 0.0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        epoch_val_acc = correct / total
        epoch_val_loss = val_loss / len(val_loader.dataset)
        
        print(f"Epoch {epoch+1}: Train Loss: {epoch_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
        
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(epoch_val_loss)
        else:
            scheduler.step()
            
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            best_epoch = epoch + 1
            
    return best_acc, best_epoch



In [10]:
def train_full_data_final(model, full_loader, criterion, optimizer, scheduler, device, epochs):
    """Final training on all data"""
    model.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        loop = tqdm(full_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)
        
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            loop.set_description(f"Loss: {loss.item():.4f}")

        epoch_loss = running_loss / len(full_loader.dataset)
        
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(epoch_loss)
        else:
            scheduler.step()
            
    torch.save(model.state_dict(), 'final_resnet_full_data.pth')


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_candidates = ['18', '50']
results = {}

# for size in model_candidates:
#     print(f"\nEvaluating ResNet-{size}")
#     model = get_resnet_scratch(size=size, num_classes=17, pretrained=True).to(device)
#     if torch.cuda.device_count() > 1:
#         print(f"Success: Found {torch.cuda.device_count()} GPUs! Training in parallel.")
#         model = nn.DataParallel(model)
#     criterion = nn.CrossEntropyLoss()
#     optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)
    
#     acc, best_ep = train_eval_split(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=8)
    
#     results[size] = {'acc': acc, 'best_epoch': best_ep}
#     print(f"ResNet-{size} Result: Best Val Acc = {acc:.4f} at Epoch {best_ep}")

# best_size = max(results, key=lambda x: results[x]['acc'])
# best_params = results[best_size]
# print(f"\nWINNER: ResNet-{best_size} with Acc {best_params['acc']:.4f}")

# final_model = get_resnet_scratch(size=best_size, num_classes=17, pretrained=True).to(device)

# if torch.cuda.device_count() > 1:
#     final_model = nn.DataParallel(final_model)

# full_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_tf) 

# targets = full_dataset.targets
# class_counts = Counter(targets)
# class_weights = {cls: 1.0/count for cls, count in class_counts.items()}
# sample_weights = [class_weights[t] for t in targets]
# sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# full_loader = DataLoader(
#     full_dataset,
#     batch_size=32,
#     sampler=sampler,
#     num_workers=2,
#     pin_memory=True
# )

# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.AdamW(final_model.parameters(), lr=1e-4, weight_decay=1e-2)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

# optimal_epochs = best_params['best_epoch'] + 1

# train_full_data_final(
#     final_model, 
#     full_loader, 
#     criterion, 
#     optimizer, 
#     scheduler, 
#     device, 
#     epochs=optimal_epochs
# )

In [12]:
# notebook_login()
repo_id = "mostafafaheem/ResNet" 

# api = HfApi()

# api.create_repo(repo_id=repo_id, exist_ok=True)

# api.upload_file(
#     path_or_fileobj="/kaggle/working/final_resnet_full_data.pth",
#     path_in_repo="model.pth",
#     repo_id=repo_id,
#     repo_type="model"
# )

In [13]:
FILENAME = "model.pth"
MODEL_SIZE = "50"
NUM_CLASSES = 17
REPO_ID = repo_id
def load_resnet_from_hub(model_size, num_classes, repo_id, filename):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = get_resnet_scratch(size=model_size, num_classes=num_classes, pretrained=False)
    
    print(f"Downloading weights from {repo_id}")
    try:
        model_path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="model")
    except Exception as e:
        print("Error downloading: Ensure your Repo ID and Filename are correct.")
        raise e

    state_dict = torch.load(model_path, map_location="cpu")
    
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k.replace("module.", "")
        new_state_dict[name] = v
        
    model.load_state_dict(new_state_dict)
    model.to(device)
    model.eval()
    print("Model Loaded Successfully!")
    return model, device


test_tf = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD) 
])

class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.filenames = [f for f in os.listdir(root_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = os.path.join(self.root_dir, filename)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):

            print(f"Warning: Corrupted image found: {filename}. Using black image.")
            image = Image.new('RGB', (224, 224), color='black')
        
        if self.transform:
            image = self.transform(image)        
            
        return image, filename

model, device = load_resnet_from_hub(MODEL_SIZE, NUM_CLASSES, REPO_ID, FILENAME)

test_dataset = TestDataset(TEST_DIR, transform=test_tf)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

all_filenames = []
all_predictions = []

with torch.no_grad():
    for images, filenames in tqdm(test_loader):
        images = images.to(device)
        
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_filenames.extend(filenames)
        all_predictions.extend(preds.cpu().numpy())

df = pd.DataFrame({
    'ImageName': all_filenames,    
    'ClassLabel': all_predictions 
})
df.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")

model.pth:   0%|          | 0.00/94.5M [00:00<?, ?B/s]

Model Loaded Successfully!


  3%|▎         | 5/172 [00:01<00:42,  3.93it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 31%|███       | 53/172 [00:15<00:29,  3.98it/s]

 32%|███▏      | 55/172 [00:16<00:38,  3.03it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 34%|███▍      | 59/172 [00:17<00:36,  3.06it/s]

 46%|████▌     | 79/172 [00:23<00:27,  3.43it/s]

 77%|███████▋  | 132/172 [00:40<00:13,  3.06it/s]

100%|██████████| 172/172 [00:53<00:00,  3.23it/s]

Submission saved to submission.csv
